# Capstone: Ridge vs. Learned Features vs. a Fixed Kernel

Capstone notebook for the [From Rules to Weights](https://josephausterweil.github.io/probintro/deep/from-rules-to-weights/) chapter (Part VIII).

**The experiment.** Part VII's models fix the feature space; the chapter's 2-4-1 network learns it. Here you run the trade head-to-head on bento data, with three contestants:

- **(a) Ridge regression** on the two hand-designed features (weight, crunch) — a Gaussian prior + MAP estimate.
- **(b) The 2-4-1 network** from the chapter — learned features, trained by gradient descent (maximum likelihood).
- **(c) Kernel ridge regression with an RBF kernel** — a fixed-kernel method standing in for the NNGP limit (exact Bayes, frozen features).

You will compare **test error as the training set grows** from 6 bentos to 200, and answer: who wins with tiny data? whose advantage grows? when (if ever) do learned features beat the hand rule here — and why is that a fair question to ask of only *two* features?

The setup and data cells below are complete and runnable. The model fits are `TODO`s — each stub states its formula. Work top to bottom.

## 0. Setup (complete — run me)

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt

key = jr.PRNGKey(0)
print('JAX ready:', jax.__version__)

## 1. Data (complete — run me)

A generative process for bentos, matching the book's canon: students bring tonkatsu about **70%** of the time; tonkatsu weighs about **500 g** (crunchy), hamburger about **350 g** (soft). Features are in the chapter's rescaled units — hundreds of grams, crunch 0–10. Labels: `1.0` = tonkatsu, `0.0` = not.

In [ ]:
def make_data(key, n):
    """Sample n bentos: X is (n, 2) = [weight in 100g, crunch 0-10], y is (n,)."""
    k1, k2, k3 = jr.split(key, 3)
    is_tonkatsu = jr.bernoulli(k1, 0.7, (n,))                  # the 70/30 base rate
    weight = jnp.where(is_tonkatsu,
                       5.0 + 0.5 * jr.normal(k2, (n,)),         # tonkatsu ~ 500 g
                       3.5 + 0.4 * jr.normal(k2, (n,)) + 0.6)   # hamburger ~ 350 g (+ overlap)
    crunch = jnp.where(is_tonkatsu,
                       8.0 + 1.2 * jr.normal(k3, (n,)),
                       3.0 + 1.2 * jr.normal(k3, (n,)))
    X = jnp.stack([weight, crunch], axis=1)
    y = is_tonkatsu.astype(jnp.float32)
    return X, y

# A fixed, generous test set -- every contestant is judged on the SAME bentos.
X_test, y_test = make_data(jr.PRNGKey(999), 500)

def test_error(predict_fn):
    """Mean squared error of a predictor on the held-out test set."""
    preds = predict_fn(X_test)
    return float(jnp.mean((preds - y_test) ** 2))

X6, y6 = make_data(jr.PRNGKey(1), 6)
print('example training draw (n=6):')
print(X6)
print('labels:', y6)

## 2. (a) Ridge on the hand features — TODO

Fit ridge regression on inputs augmented with a constant-1 column (so the offset $b$ is learned too). Closed form:

$$\hat{\mathbf{w}} = (\Phi^\top \Phi + \lambda I)^{-1} \Phi^\top \mathbf{y}, \qquad \Phi = [\mathbf{X},\, \mathbf{1}].$$

Use `jnp.linalg.solve` rather than an explicit inverse. Start with `lam = 0.1`. Remember what $\lambda$ *is*: $\sigma^2/\tau^2$, the Gaussian prior in disguise (bias-variance chapter).

In [ ]:
def ridge_fit(X, y, lam=0.1):
    """Return a predict(Xnew) function fit by ridge on [X, 1]."""
    # TODO:
    #   1. Phi = jnp.concatenate([X, jnp.ones((X.shape[0], 1))], axis=1)
    #   2. w_hat = solve(Phi.T @ Phi + lam * I, Phi.T @ y)
    #   3. return lambda Xnew: [Xnew, 1] @ w_hat
    raise NotImplementedError

# When ridge_fit works, this should print a small number (roughly 0.05-0.15):
# print('ridge test error (n=6):', test_error(ridge_fit(X6, y6)))

## 3. (b) The 2-4-1 network — TODO

Port the chapter's training cell into a function. Same architecture (2 inputs → 4 ReLU features → 1 output), same loss (mean squared error = Gaussian NLL), same optimizer (plain gradient descent, `lr = 0.01`). Train for `steps = 500` here — more data can use more steps. Initialize from the `key` argument so each call is reproducible.

In [ ]:
def net_fit(X, y, key, steps=500, lr=0.01):
    """Return a predict(Xnew) function: a 2-4-1 ReLU net trained by GD."""
    def net(params, x):
        h = jnp.maximum(0.0, params['A'] @ x + params['c'])
        return jnp.dot(params['w'], h) + params['b']
    # TODO:
    #   1. loss(params) = mean squared error of jax.vmap(net) over (X, y)
    #   2. init params: A ~ 0.5*normal(k1, (4,2)), c = zeros(4),
    #                   w ~ 0.5*normal(k2, (4,)), b = 0.0
    #   3. loop: params = jax.tree.map(lambda p, g: p - lr*g, params, grad(loss)(params))
    #   4. return lambda Xnew: jax.vmap(lambda x: net(params, x))(Xnew)
    raise NotImplementedError

# When net_fit works:
# print('net test error (n=6):', test_error(net_fit(X6, y6, jr.PRNGKey(0))))

## 4. (c) Kernel ridge with an RBF kernel — TODO

The fixed-kernel contestant (the NNGP limit's stand-in): no features are learned; a kernel declares similarity in advance and prediction is a similarity-weighted vote (Gaussian-processes chapter):

$$k(\mathbf{x}, \mathbf{x}') = \exp\!\left(-\frac{\lVert\mathbf{x}-\mathbf{x}'\rVert^2}{2\ell^2}\right), \qquad \hat{f}(\mathbf{x}_*) = k(\mathbf{x}_*, X)\,(K + \lambda I)^{-1}\,\mathbf{y}.$$

Start with length-scale `ell = 1.5` and `lam = 0.1`. Recall $\lambda$ here *is* the GP's noise variance $\sigma_n^2$ — the same seam as ridge.

In [ ]:
def rbf_kernel(Xa, Xb, ell=1.5):
    """Kernel matrix K[i, j] = exp(-||Xa_i - Xb_j||^2 / (2 ell^2))."""
    # TODO: sq_dists via broadcasting, e.g. Xa[:, None, :] - Xb[None, :, :]
    raise NotImplementedError

def kernel_ridge_fit(X, y, ell=1.5, lam=0.1):
    """Return a predict(Xnew) function via kernel ridge regression."""
    # TODO: alpha = solve(rbf_kernel(X, X, ell) + lam * I, y)
    #       return lambda Xnew: rbf_kernel(Xnew, X, ell) @ alpha
    raise NotImplementedError

# When both work:
# print('kernel ridge test error (n=6):', test_error(kernel_ridge_fit(X6, y6)))

## 5. (d) The race: test error as data grows — TODO

For each training-set size `n` in `[6, 12, 25, 50, 100, 200]`, draw a fresh training set (`jr.fold_in(key, n)` keeps it reproducible), fit all three contestants, and record each one's `test_error`. Plot the three curves (log-x helps).

**Questions to answer in a sentence or two each:**
1. Who wins at `n = 6`, and why does a strong prior (ridge's Gaussian, the kernel's smoothness) matter most exactly there?
2. Does the learned-features net ever *beat* the fixed-feature methods here? Why is two clean hand features a best-case scenario for ridge — and what input (hint: the kiosk's) would flip the result?
3. Where do the kernel method and the net converge, and what does that say about the chapter's "fixed kernel vs. learned features" trade?

In [ ]:
ns = [6, 12, 25, 50, 100, 200]

# TODO: loop over ns, fit (a), (b), (c) on make_data(jr.fold_in(key, n), n),
#       collect test errors, then plot:
#
# plt.figure(figsize=(7, 4.5))
# plt.plot(ns, errs_ridge,  'o-', label='ridge (hand features)')
# plt.plot(ns, errs_net,    's-', label='2-4-1 net (learned features)')
# plt.plot(ns, errs_kernel, '^-', label='kernel ridge (RBF, fixed kernel)')
# plt.xscale('log'); plt.xlabel('training set size n'); plt.ylabel('test MSE')
# plt.legend(); plt.title('One question, three answers, one race')
# plt.show()

## 6. Optional stretch: double descent (Belkin et al., 2019)

Fix `n = 25`. Sweep the network's hidden width `H` from 1 to well past `n` (say `[1, 2, 4, 8, 16, 32, 64, 128]`), train each width (a few random restarts help), and plot test error vs. `H`. Look for the shape from the bias-variance chapter: error falling, spiking near the interpolation threshold, then *descending again* as the model becomes heavily over-parameterized. In the chapter's language: capacity taken to infinity doesn't escape having a prior — it becomes one (the NNGP).

*Reference: Belkin, Hsu, Ma, & Mandal (2019), "Reconciling modern machine-learning practice and the classical bias-variance trade-off," PNAS 116(32).*

In [ ]:
# TODO (optional): generalize net_fit to take a hidden width H, then sweep it.
# def net_fit_width(X, y, key, H, steps=500, lr=0.01):
#     ...

---

*This project is generously funded by the Japanese Probabilistic Computing Consortium Association ([JPCCA](https://jpcca.org/)).*